# Challenge 05: Data May Be Duplicated or Missing

**PPT problem:** Data may be duplicated or missing  
**Technique:** Deduplication / data quality checks

A job can complete successfully and still produce wrong data. Duplicate rows can appear because of API behavior, retry logic, or reruns. A data engineer should identify a stable key and check whether row counts match unique key counts.


## Setup

Make sure the mock serving API is running:

```bash
cd day_2/day_2_1_API
python -m uvicorn mock_serving_api:app --reload --port 8011
```

If you use a different local port, update `BASE_URL` in the code cell.


## Run the Broken Program First

Do not fix the code before running it. Run it once and observe the failure signal.

Look for:

- Does the script raise an exception?
- Do rows collected and unique record_id values match?
- Which column should be the unique key?



In [ ]:
"""
Challenge 05: Duplicate records.

The script runs without an exception, but the result is wrong. The API returns
some duplicate records. Fix the client so each record_id is counted once.
"""

import pandas as pd
import requests


BASE_URL = "http://127.0.0.1:8011"
CLIENT_ID = "group_05_duplicates"


def collect_all_pages() -> list[dict]:
    page = 1
    records = []

    while True:
        response = requests.get(
            f"{BASE_URL}/debug-lab/05-duplicates/taxi/records",
            params={
                "page": page,
                "page_size": 20,
                "client_id": CLIENT_ID,
            },
            timeout=10,
        )
        response.raise_for_status()
        payload = response.json()
        records.extend(payload["data"])

        if payload["next_page"] is None:
            break
        page = payload["next_page"]

    return records


def main() -> None:
    records = collect_all_pages()
    dataframe = pd.DataFrame(records)

    print("Rows collected:", len(dataframe))
    print("Unique record_id values:", dataframe["record_id"].nunique())
    print("Expected: these two numbers should match.")
    print(dataframe[["record_id", "longitude", "latitude"]].head())


if __name__ == "__main__":
    main()


## Diagnose

Write short answers before changing the code.

1. Did the HTTP request fail, or did the completed job produce a wrong result?
2. What row counts show that duplicate records exist?
3. Which column should be treated as the stable unique key?
4. Which technique from the PPT table should fix it?


In [ ]:
# Notes
# 1.
# 2.
# 3.
# 4.


## Where to Look for the Fix

Use the API docs, the observed failure, and the clues below to decide what to change.

Primary place to check: `http://127.0.0.1:8011/docs`, then find `/debug-lab/05-duplicates/taxi/records`.

Use these clues:

1. The request may succeed even when the data is wrong.
2. Compare row count with unique `record_id` count.
3. Use `record_id` as the deduplication key before computing summaries.

After that, use the success condition below to check whether your repaired code is good enough.


## Repair Workspace

Copy the broken code here and edit it until it works.

Success condition: The notebook should keep one row per `record_id`, and the row count should match the unique `record_id` count.


In [ ]:
# Paste the broken code here, then repair it.
# Start by checking /docs for the endpoint contract and rereading the error output above.
# Stop when the success condition in the previous markdown cell is satisfied.
